# WavLM Speech Emotion Recognition (SpeechBrain) — cleaned-up training notebook

**Before "Save & Run All" (right-hand Settings panel):**
1. Accelerator = **GPU T4**, Internet = **On**
2. Add data: `processed-ser-data` (training audio) and `test-data` (demo audio)
3. Add-ons → Secrets: `HF_TOKEN` attached
4. Check the `EPOCHS` / `TIME_BUDGET_HOURS` cell below fits your session limit

**What changed vs the previous version**

| Area | Change |
|---|---|
| Setup | Pinned installs run first; unused `wavlm-large` baseline removed; one config cell |
| Checkpoints | Isolated under `RUN_TAG/` in the HF repo, so the old run is never touched. Stale remote folders are pruned by diff (the old code could delete the checkpoint it had just uploaded) |
| Training | Differential LR (backbone 1e-5, head 1e-3), 6 s random crop, batch 8 x accumulation 2, SNR-based noise, fp32 pooling/loss, full seeding |
| Metrics | Model selection on **UAR** (mean per-class recall); macro-F1, accuracy, ECE reported |
| Parity | Evaluation and inference use the *same* SpeechBrain modules as training (the old HF-based eval skipped `output_norm`) |
| Safety | Time budget stops training gracefully so the test and export cells still run |
| Data | Sorted/deterministic file scan, header-only duration read, optional held-out corpus for cross-corpus testing |

**Note:** manifests (`train/valid/test.json`) are created on the first run and then reused (also when resumed from the Hub). To change `HOLDOUT_DATASET`, use a new `RUN_TAG`.

In [ ]:
!pip install -q "speechbrain==1.0.0" "huggingface_hub==0.32.0" "transformers==4.52.4" soundfile

In [ ]:
import os, sys, shutil, json, time
from pathlib import Path
import torch

# ------------------------- edit here ---------------------------------------
SEED              = 1986
HF_REPO_ID        = "space-cadet/wavlm-ser"
RUN_TAG           = "v2"       # checkpoints live under <repo>/<RUN_TAG>/ ; change to start a new experiment
EPOCHS            = 20         # upper bound; early stopping / time budget may end sooner
TIME_BUDGET_HOURS = 8.0        # stop after the epoch that crosses this (keeps room for test + export)
HOLDOUT_DATASET   = ""         # e.g. "tess" -> that whole corpus becomes the test set (cross-corpus eval)
DATA_FOLDER       = "/kaggle/input/processed-ser-data"
TEST_AUDIO_GLOB   = "/kaggle/input/test-data/**/*.wav"
# ---------------------------------------------------------------------------

WORK = Path("/kaggle/working/emotion_classification")
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
OUT = WORK / "results" / "ser_wavlm" / str(SEED)
SAVE_FOLDER = OUT / "save"
SAVE_FOLDER.mkdir(parents=True, exist_ok=True)

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN not found. Attach it under Add-ons -> Secrets.")

os.environ.update(
    HF_TOKEN=HF_TOKEN, HF_REPO_ID=HF_REPO_ID, RUN_TAG=RUN_TAG,
    PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True",
)

if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Set Accelerator to GPU T4 in the notebook settings.")
print("GPU:", torch.cuda.get_device_name(0))
print("torch", torch.__version__, "| work dir:", WORK)

## 1. Restore checkpoint (resume support)

In [ ]:
from huggingface_hub import HfApi, hf_hub_download, login

login(token=HF_TOKEN)
api = HfApi()
api.create_repo(HF_REPO_ID, repo_type="model", exist_ok=True, private=True)

prefix = f"{RUN_TAG}/"
remote = [f for f in api.list_repo_files(HF_REPO_ID, repo_type="model")
          if f.startswith(prefix) and "pretrained_models/" not in f and "_incompatible/" not in f]

if not remote:
    print(f"Nothing under '{prefix}' -> fresh run.")
else:
    tmp = WORK / "_hf_dl"
    for f in remote:
        p = hf_hub_download(HF_REPO_ID, f, repo_type="model", local_dir=str(tmp))
        dest = SAVE_FOLDER / f[len(prefix):]
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(p, dest)
        print("  restored", f[len(prefix):])
    shutil.rmtree(tmp, ignore_errors=True)
    print("Checkpoint restored; training will resume.")

## 2. Code: shared helpers, data prep, config, training script

In [ ]:
%%writefile /kaggle/working/emotion_classification/ser_common.py
import os
import random

import numpy as np
import torch
import torchaudio


class MLPClassifier(torch.nn.Module):
    def __init__(self, input_size, hidden=256, n_classes=8, dropout=0.3):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(input_size, hidden),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(hidden, n_classes),
        )

    def forward(self, x):
        if x.dim() == 3:        # [B, 1, D] from StatisticsPooling
            x = x.squeeze(1)
        return self.net(x)


def set_seed(seed, deterministic=False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = deterministic
    torch.backends.cudnn.benchmark = not deterministic


def load_wav(path, sample_rate=16000):
    # mono float32 [T] at the target sample rate
    sig, sr = torchaudio.load(path)
    sig = sig.mean(dim=0)
    if sr != sample_rate:
        sig = torchaudio.functional.resample(sig, sr, sample_rate)
    return sig


def expected_calibration_error(conf, correct, n_bins=15):
    conf = np.asarray(conf, dtype=np.float64)
    correct = np.asarray(correct, dtype=np.float64)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece, n = 0.0, len(conf)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            ece += m.sum() / n * abs(correct[m].mean() - conf[m].mean())
    return float(ece)


def hf_context():
    repo = os.environ.get("HF_REPO_ID", "").strip()
    token = os.environ.get("HF_TOKEN")
    tag = os.environ.get("RUN_TAG", "v2").strip("/")
    if not repo or not token:
        return None
    from huggingface_hub import HfApi
    return HfApi(), repo, tag, token

In [ ]:
%%writefile /kaggle/working/emotion_classification/prepare_data.py
#!/usr/bin/env python3
# Builds train/valid/test JSON manifests from files named  dataset_emotion_number.wav
import json
import logging
import sys
from collections import Counter
from pathlib import Path

import soundfile as sf
from sklearn.model_selection import train_test_split
from tqdm import tqdm

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

VALID_EMOTIONS = {"angry", "happy", "sad", "neutral", "fearful", "disgust", "surprised", "calm"}
VALID_DATASETS = {"ravdess", "cremad", "iemocap", "savee", "emo-db", "tess"}
EMOTION_MAPPING = {
    "fru": "angry", "frustration": "angry",
    "exc": "happy", "excitement": "happy",
    "boredom": "neutral", "bor": "neutral", "oth": "neutral",
    "xxx": None,
}


def parse_filename(path):
    parts = path.stem.lower().split("_")
    if len(parts) < 3:
        return None
    dataset, emotion = parts[0], parts[1]
    if dataset not in VALID_DATASETS:
        dataset = next((d for d in sorted(VALID_DATASETS) if d in dataset or dataset in d), None)
        if dataset is None:
            return None
    if emotion not in VALID_EMOTIONS:
        emotion = EMOTION_MAPPING.get(emotion)
        if emotion is None:
            return None
    return dataset, emotion


def scan(data_folder, min_dur=0.1, max_dur=30.0):
    files = sorted(set(data_folder.glob("*.wav")) | set(data_folder.glob("*.WAV")))
    if not files:
        raise RuntimeError(f"No .wav files in {data_folder}")
    logger.info("Found %d audio files", len(files))
    items, skipped = [], Counter()
    for i, f in enumerate(tqdm(files, desc="Scanning", disable=not sys.stderr.isatty())):
        meta = parse_filename(f)
        if meta is None:
            skipped["bad_filename"] += 1
            continue
        try:
            info = sf.info(str(f))           # header only, no decoding
        except Exception:
            skipped["unreadable"] += 1
            continue
        dur = info.frames / float(info.samplerate)
        if not (min_dur <= dur <= max_dur):
            skipped["duration_out_of_range"] += 1
            continue
        items.append({
            "id": f"utt-{i:06d}", "wav": str(f.resolve()),
            "dataset": meta[0], "emotion": meta[1],
            "duration": round(dur, 3), "sample_rate": info.samplerate,
        })
    logger.info("Kept %d files; skipped: %s", len(items), dict(skipped))
    return items


def _strat_split(data, test_size, seed):
    try:
        return train_test_split(data, test_size=test_size,
                                stratify=[d["emotion"] for d in data], random_state=seed)
    except ValueError as e:
        logger.warning("Stratified split failed (%s); using random split", e)
        return train_test_split(data, test_size=test_size, random_state=seed)


def split_items(items, ratio, seed, holdout_dataset=None):
    tr, va, te = ratio
    if holdout_dataset:
        test = [d for d in items if d["dataset"] == holdout_dataset]
        rest = [d for d in items if d["dataset"] != holdout_dataset]
        if not test:
            raise ValueError(f"holdout_dataset '{holdout_dataset}' has no files")
        train, valid = _strat_split(rest, va / (tr + va), seed)
        logger.info("Cross-corpus mode: '%s' is the test set (%d clips)", holdout_dataset, len(test))
        return train, valid, test
    logger.warning("Random stratified split: speakers overlap across splits, so scores are "
                   "in-distribution. Use holdout_dataset for a cross-corpus estimate.")
    train, temp = _strat_split(items, (va + te) / 100.0, seed)
    valid, test = _strat_split(temp, te / float(va + te), seed)
    return train, valid, test


def prepare_data(data_folder, save_folder, split_ratio=(80, 10, 10), holdout_dataset=None,
                 seed=42, force=False, **_ignored):
    data_folder = Path(data_folder).resolve()
    save_path = Path(save_folder).resolve() / "save"
    save_path.mkdir(parents=True, exist_ok=True)
    outs = {n: save_path / f"{n}.json" for n in ("train", "valid", "test")}
    if not force and all(p.exists() for p in outs.values()):
        logger.info("Manifests already exist in %s (reusing; force=True to rebuild)", save_path)
        return True

    items = scan(data_folder)
    if not items:
        raise RuntimeError("No valid samples. Check the filename format.")
    train, valid, test = split_items(items, split_ratio, seed, holdout_dataset or None)

    meta = {"seed": seed, "holdout_dataset": holdout_dataset or None, "splits": {}}
    for name, split in (("train", train), ("valid", valid), ("test", test)):
        with open(outs[name], "w") as f:
            json.dump({d["id"]: {"wav": d["wav"], "emotion": d["emotion"], "dataset": d["dataset"]}
                       for d in split}, f, indent=1)
        meta["splits"][name] = {
            "n": len(split),
            "emotions": dict(Counter(d["emotion"] for d in split)),
            "datasets": dict(Counter(d["dataset"] for d in split)),
            "hours": round(sum(d["duration"] for d in split) / 3600, 2),
        }
        logger.info("%s: %d clips | %s", name, len(split), meta["splits"][name]["emotions"])
    with open(save_path / "split_meta.json", "w") as f:
        json.dump(meta, f, indent=2)
    return True

In [ ]:
%%writefile /kaggle/working/emotion_classification/train.yaml
seed: 1986
__set_seed: !apply:ser_common.set_seed [!ref <seed>]
output_folder: !ref results/ser_wavlm/<seed>
save_folder: !ref <output_folder>/save
train_log: !ref <output_folder>/train_log.txt

# -- data ---------------------------------------------------------------
data_folder: /kaggle/input/processed-ser-data/
holdout_dataset: null
label_encoder_file: !ref <save_folder>/label_encoder.txt
train_json: !ref <save_folder>/train.json
valid_json: !ref <save_folder>/valid.json
test_json: !ref <save_folder>/test.json
sample_rate: 16000
max_train_sec: 6.0      # random crop at train time (bounds O(T^2) attention cost)
max_eval_sec: 8.0       # head crop at valid/test/inference time

# -- model --------------------------------------------------------------
pooled_dim: 1536        # 2 x 768 (mean + std)
n_classes: 8
hidden_dim: 256
dropout: 0.3

wavlm: !new:speechbrain.lobes.models.huggingface_transformers.wavlm.WavLM
    source: microsoft/wavlm-base-plus
    save_path: pretrained_models/wavlm
    output_norm: True
    freeze: False
    freeze_feature_extractor: True

pooling: !new:speechbrain.nnet.pooling.StatisticsPooling

classifier: !new:ser_common.MLPClassifier
    input_size: !ref <pooled_dim>
    hidden: !ref <hidden_dim>
    n_classes: !ref <n_classes>
    dropout: !ref <dropout>

modules:
    wavlm: !ref <wavlm>
    pooling: !ref <pooling>
    classifier: !ref <classifier>

# -- optimisation -------------------------------------------------------
number_of_epochs: 20
batch_size: 8
eval_batch_size: 16
grad_accumulation_factor: 2     # effective batch = 16
max_grad_norm: 5.0
precision: fp16
label_smoothing: 0.0

lr_backbone: 0.00001
lr_head: 0.001
weight_decay: 0.05
opt_class: !name:torch.optim.AdamW
    weight_decay: !ref <weight_decay>

# NewBob is used as a *multiplier* on both base LRs (initial_value = 1.0)
lr_annealing: !new:speechbrain.nnet.schedulers.NewBobScheduler
    initial_value: 1.0
    improvement_threshold: 0.0025
    annealing_factor: 0.8
    patient: 2

early_stop_patience: 5
early_stop_min_delta: 0.001
time_budget_hours: 8.0

# -- augmentation (train only) ------------------------------------------
speed_factors: [0.9, 1.0, 1.1]
noise_prob: 0.5
noise_snr_db: [10.0, 30.0]

# -- bookkeeping --------------------------------------------------------
train_logger: !new:speechbrain.utils.train_logger.FileTrainLogger
    save_file: !ref <train_log>

train_dataloader_opts:
    batch_size: !ref <batch_size>
    shuffle: True
    num_workers: 2
    drop_last: True

valid_dataloader_opts:
    batch_size: !ref <eval_batch_size>
    num_workers: 2

test_dataloader_opts:
    batch_size: !ref <eval_batch_size>
    num_workers: 2

epoch_counter: !new:speechbrain.utils.epoch_loop.EpochCounter
    limit: !ref <number_of_epochs>

checkpointer: !new:speechbrain.utils.checkpoints.Checkpointer
    checkpoints_dir: !ref <save_folder>
    recoverables:
        wavlm: !ref <wavlm>
        classifier: !ref <classifier>
        counter: !ref <epoch_counter>
        lr_annealing: !ref <lr_annealing>

In [ ]:
%%writefile /kaggle/working/emotion_classification/train.py
#!/usr/bin/env python3
import json
import logging
import os
import random
import shutil
import sys
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import speechbrain as sb
import torch
import torchaudio
from hyperpyyaml import load_hyperpyyaml
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
                             f1_score, precision_recall_fscore_support)
from speechbrain.utils.distributed import run_on_main

from ser_common import expected_calibration_error, hf_context, load_wav

logger = logging.getLogger(__name__)
T0 = time.time()


class EmotionBrain(sb.core.Brain):

    # ---- optimiser: two parameter groups (backbone / head) ---------------
    def init_optimizers(self):
        h = self.hparams
        backbone = [p for p in self.modules.wavlm.parameters() if p.requires_grad]
        head = list(self.modules.classifier.parameters())
        self.optimizer = h.opt_class([
            {"params": backbone, "lr": h.lr_backbone},
            {"params": head, "lr": h.lr_head},
        ])
        self.base_lrs = [h.lr_backbone, h.lr_head]
        if self.checkpointer is not None:
            self.checkpointer.add_recoverable("optimizer", self.optimizer)

    def on_fit_start(self):
        # optimiser creation and checkpoint recovery are done in main()
        if not hasattr(self, "optimizer"):
            self.init_optimizers()

    # ---- augmentation ----------------------------------------------------
    def augment(self, wavs):
        h = self.hparams
        f = random.choice(h.speed_factors)
        if f != 1.0:
            wavs = torchaudio.functional.resample(
                wavs, orig_freq=int(round(h.sample_rate * f)), new_freq=h.sample_rate)
        if random.random() < h.noise_prob:
            snr_db = random.uniform(*h.noise_snr_db)
            sig_pow = wavs.pow(2).mean(dim=-1, keepdim=True).clamp_min(1e-10)
            noise_pow = sig_pow / (10.0 ** (snr_db / 10.0))
            wavs = wavs + torch.randn_like(wavs) * noise_pow.sqrt()
        return wavs

    # ---- forward / loss ---------------------------------------------------
    def compute_forward(self, batch, stage):
        batch = batch.to(self.device)
        wavs, wav_lens = batch.sig
        if stage == sb.Stage.TRAIN:
            wavs = self.augment(wavs)
        feats = self.modules.wavlm(wavs)
        if isinstance(feats, dict):
            feats = feats["last_hidden_state"]
        feats = feats.float()                       # pooling statistics in fp32
        pooled = self.modules.pooling(feats, wav_lens)
        return self.modules.classifier(pooled)

    def compute_objectives(self, logits, batch, stage):
        emo, _ = batch.emo_encoded
        targets = emo.long().view(-1).to(logits.device)
        logits = logits.float()
        loss = torch.nn.functional.cross_entropy(
            logits, targets, weight=self.class_weights,
            label_smoothing=self.hparams.label_smoothing)
        if stage != sb.Stage.TRAIN:
            conf, pred = logits.softmax(dim=-1).max(dim=-1)
            self._y_true.append(targets.cpu())
            self._y_pred.append(pred.cpu())
            self._conf.append(conf.cpu())
            self._ids.extend(list(batch.id))
        return loss

    # ---- stage hooks --------------------------------------------------------
    def on_stage_start(self, stage, epoch=None):
        if stage != sb.Stage.TRAIN:
            self._y_true, self._y_pred, self._conf, self._ids = [], [], [], []

    def _metrics(self):
        y_true = torch.cat(self._y_true).numpy()
        y_pred = torch.cat(self._y_pred).numpy()
        labels = list(range(len(self.class_names)))
        uar = balanced_accuracy_score(y_true, y_pred)
        return y_true, y_pred, {
            "acc": accuracy_score(y_true, y_pred),
            "uar": uar,
            "macro_f1": f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0),
            "uar_error": 1.0 - uar,
        }

    def on_stage_end(self, stage, stage_loss, epoch=None):
        if stage == sb.Stage.TRAIN:
            self.train_loss = stage_loss
            return

        y_true, y_pred, m = self._metrics()
        m["loss"] = stage_loss

        if stage == sb.Stage.VALID:
            old_mult, new_mult = self.hparams.lr_annealing(m["uar_error"])
            for g, base in zip(self.optimizer.param_groups, self.base_lrs):
                g["lr"] = base * new_mult
            print(f"epoch {epoch:02d} | train {self.train_loss:.4f} | valid {m['loss']:.4f} | "
                  f"acc {m['acc']:.4f} | UAR {m['uar']:.4f} | macroF1 {m['macro_f1']:.4f} | "
                  f"lr x{old_mult:.3f}->x{new_mult:.3f} | {(time.time() - T0) / 3600:.2f} h", flush=True)

            self.hparams.train_logger.log_stats(
                stats_meta={"epoch": epoch, "lr_backbone": self.base_lrs[0] * old_mult,
                            "lr_head": self.base_lrs[1] * old_mult},
                train_stats={"loss": self.train_loss}, valid_stats=m)

            self.checkpointer.save_and_keep_only(
                meta={k: float(v) for k, v in m.items()}, min_keys=["uar_error"])
            self._push_to_hub(epoch, m)

            # early stopping + wall-clock budget
            best = getattr(self, "_best", float("inf"))
            if m["uar_error"] < best - self.hparams.early_stop_min_delta:
                self._best, self._bad = m["uar_error"], 0
            else:
                self._bad = getattr(self, "_bad", 0) + 1
                print(f"  no improvement {self._bad}/{self.hparams.early_stop_patience} "
                      f"(best UAR error {best:.4f})")
            stop = self._bad >= self.hparams.early_stop_patience
            if (time.time() - T0) / 3600 >= self.hparams.time_budget_hours:
                print("  time budget reached -> stopping after this epoch")
                stop = True
            if stop:
                self.hparams.epoch_counter.current = self.hparams.epoch_counter.limit

        elif stage == sb.Stage.TEST:
            self._write_test_report(y_true, y_pred, m)

    # ---- test report -------------------------------------------------------
    def _write_test_report(self, y_true, y_pred, m):
        names = list(self.class_names)
        labels = list(range(len(names)))
        p, r, f, s = precision_recall_fscore_support(
            y_true, y_pred, labels=labels, zero_division=0)
        cm = confusion_matrix(y_true, y_pred, labels=labels)
        conf = torch.cat(self._conf).numpy()
        ece = expected_calibration_error(conf, y_true == y_pred)
        report = {
            "class_names": names, "n": int(len(y_true)),
            "accuracy": float(m["acc"]), "uar": float(m["uar"]),
            "macro_f1": float(m["macro_f1"]), "loss": float(m["loss"]), "ece": ece,
            "per_class": {n: {"precision": float(p[i]), "recall": float(r[i]),
                              "f1": float(f[i]), "support": int(s[i])}
                          for i, n in enumerate(names)},
            "confusion_matrix": cm.tolist(),
            "holdout_dataset": self.hparams.holdout_dataset,
        }
        out = Path(self.hparams.output_folder)
        with open(out / "test_report.json", "w") as fh:
            json.dump(report, fh, indent=2)
        pd.DataFrame({
            "id": self._ids,
            "true": np.array(names)[y_true], "pred": np.array(names)[y_pred],
            "confidence": conf,
        }).to_csv(out / "test_predictions.csv", index=False)
        print(f"TEST | acc {m['acc']:.4f} | UAR {m['uar']:.4f} | macroF1 {m['macro_f1']:.4f} | ECE {ece:.4f}")

    # ---- HF Hub push (never deletes what it just uploaded) --------------------
    def _push_to_hub(self, epoch, m):
        ctx = hf_context()
        if ctx is None:
            return
        api, repo, tag, token = ctx
        try:
            save_dir = Path(self.hparams.save_folder)
            log_src = Path(self.hparams.train_log)
            if log_src.exists():
                shutil.copy2(log_src, save_dir / "train_log.txt")
            local = {p.name for p in save_dir.glob("CKPT+*") if p.is_dir()}
            api.upload_folder(
                folder_path=str(save_dir), path_in_repo=tag, repo_id=repo, repo_type="model",
                token=token, ignore_patterns=["pretrained_models/**", "_incompatible/**"],
                commit_message=f"Epoch {epoch} | UAR {m['uar']:.4f} | macroF1 {m['macro_f1']:.4f}")
            prefix = f"{tag}/"
            remote = {f[len(prefix):].split("/")[0]
                      for f in api.list_repo_files(repo, repo_type="model", token=token)
                      if f.startswith(prefix + "CKPT+")}
            for stale in sorted(remote - local):
                api.delete_folder(path_in_repo=f"{tag}/{stale}", repo_id=repo, repo_type="model",
                                  token=token, commit_message=f"Prune superseded checkpoint {stale}")
            print(f"  pushed epoch {epoch} to {repo}/{tag}", flush=True)
        except Exception as e:
            print(f"  HF push failed (epoch {epoch}): {e}", flush=True)


def recover_or_restart(brain):
    # Resume if the checkpoint matches; otherwise quarantine it locally and restart cleanly.
    try:
        brain.checkpointer.recover_if_possible()
        cur = brain.hparams.epoch_counter.current
        print(f"Resuming after epoch {cur}" if cur else "No checkpoint: fresh start.")
    except (KeyError, RuntimeError, ValueError) as e:
        save = Path(brain.hparams.save_folder)
        quarantine = save / "_incompatible"
        quarantine.mkdir(exist_ok=True)
        moved = 0
        for p in sorted(save.glob("CKPT+*")):
            shutil.move(str(p), str(quarantine / p.name))
            moved += 1
        if not moved:
            raise
        print(f"Incompatible checkpoint ({str(e)[:200]}). Moved {moved} to {quarantine}; restarting.")
        os.execv(sys.executable, [sys.executable] + sys.argv)   # re-init modules from scratch


def dataio_prepare(hparams):
    sr = hparams["sample_rate"]
    max_tr = int(hparams["max_train_sec"] * sr)
    max_ev = int(hparams["max_eval_sec"] * sr)

    train_data = sb.dataio.dataset.DynamicItemDataset.from_json(json_path=hparams["train_json"])
    valid_data = sb.dataio.dataset.DynamicItemDataset.from_json(json_path=hparams["valid_json"])
    test_data = sb.dataio.dataset.DynamicItemDataset.from_json(json_path=hparams["test_json"])
    datasets = [train_data, valid_data, test_data]

    @sb.utils.data_pipeline.takes("wav")
    @sb.utils.data_pipeline.provides("sig")
    def train_audio(wav):
        sig = load_wav(wav, sr)
        if sig.numel() > max_tr:
            start = random.randint(0, sig.numel() - max_tr)
            sig = sig[start:start + max_tr]
        return sig

    @sb.utils.data_pipeline.takes("wav")
    @sb.utils.data_pipeline.provides("sig")
    def eval_audio(wav):
        return load_wav(wav, sr)[:max_ev]

    sb.dataio.dataset.add_dynamic_item([train_data], train_audio)
    sb.dataio.dataset.add_dynamic_item([valid_data, test_data], eval_audio)

    emotions = sorted({item["emotion"] for d in datasets for item in d.data.values()})
    print(f"Classes ({len(emotions)}): {emotions}")

    label_encoder = sb.dataio.encoder.CategoricalEncoder()
    for e in emotions:
        label_encoder.add_label(e)
    label_encoder.expect_len(len(emotions))

    counts = Counter(item["emotion"] for item in train_data.data.values())
    total = sum(counts.values())
    class_weights = torch.tensor(
        [total / (len(emotions) * counts.get(e, 1)) for e in emotions], dtype=torch.float)
    print("Class weights:", {e: round(float(w), 2) for e, w in zip(emotions, class_weights)})

    @sb.utils.data_pipeline.takes("emotion")
    @sb.utils.data_pipeline.provides("emotion", "emo_encoded")
    def emotion_pipeline(emotion):
        enc = label_encoder.encode_label_torch(emotion)
        if enc.dim() == 0:
            enc = enc.unsqueeze(0)
        yield emotion
        yield enc.long()

    sb.dataio.dataset.add_dynamic_item(datasets, emotion_pipeline)
    os.makedirs(os.path.dirname(hparams["label_encoder_file"]), exist_ok=True)
    label_encoder.save(hparams["label_encoder_file"])
    sb.dataio.dataset.set_output_keys(datasets, ["id", "sig", "emo_encoded"])
    return train_data, valid_data, test_data, label_encoder, class_weights


if __name__ == "__main__":
    hparams_file, run_opts, overrides = sb.parse_arguments(sys.argv[1:])
    sb.utils.distributed.ddp_init_group(run_opts)
    with open(hparams_file) as fin:
        hparams = load_hyperpyyaml(fin, overrides)

    sb.create_experiment_directory(
        experiment_directory=hparams["output_folder"],
        hyperparams_to_save=hparams_file, overrides=overrides)

    from prepare_data import prepare_data
    run_on_main(prepare_data, kwargs={
        "data_folder": hparams["data_folder"],
        "save_folder": hparams["output_folder"],
        "holdout_dataset": hparams["holdout_dataset"],
        "force": False,
    })

    train_data, valid_data, test_data, label_encoder, class_weights = dataio_prepare(hparams)
    n_cls = len(label_encoder.lab2ind)
    assert n_cls == hparams["n_classes"], f"data has {n_cls} classes, yaml says {hparams['n_classes']}"

    brain = EmotionBrain(
        modules=hparams["modules"], opt_class=hparams["opt_class"], hparams=hparams,
        run_opts=run_opts, checkpointer=hparams["checkpointer"])
    brain.class_names = [label_encoder.ind2lab[i] for i in range(n_cls)]
    brain.class_weights = class_weights.to(brain.device)

    print(f"Params: {sum(p.numel() for p in brain.modules.parameters()) / 1e6:.1f}M total, "
          f"{sum(p.numel() for p in brain.modules.parameters() if p.requires_grad) / 1e6:.1f}M trainable")

    brain.init_optimizers()          # must exist before recovery so optimizer state is restored
    recover_or_restart(brain)

    brain.fit(
        brain.hparams.epoch_counter, train_data, valid_data,
        train_loader_kwargs=hparams["train_dataloader_opts"],
        valid_loader_kwargs=hparams["valid_dataloader_opts"],
        progressbar=False)

    brain.evaluate(
        test_data, min_key="uar_error", progressbar=False,
        test_loader_kwargs=hparams["test_dataloader_opts"])

    ctx = hf_context()
    if ctx is not None:
        api, repo, tag, token = ctx
        try:
            out = Path(hparams["output_folder"])
            for name in ("test_report.json", "test_predictions.csv"):
                api.upload_file(path_or_fileobj=str(out / name), path_in_repo=f"{tag}/{name}",
                                repo_id=repo, repo_type="model", token=token)
            print("Test report pushed to HF.")
        except Exception as e:
            print(f"Could not push test report: {e}")

## 3. Train
Prints one summary line per epoch (progress bars are off to keep this notebook small). Resumes automatically from `RUN_TAG/` on the HF Hub.

In [ ]:
cmd = (f"python train.py train.yaml --seed {SEED} --number_of_epochs {EPOCHS} "
       f"--time_budget_hours {TIME_BUDGET_HOURS}")
if HOLDOUT_DATASET:
    cmd += f" --holdout_dataset {HOLDOUT_DATASET}"
print(cmd)
!{cmd}

## 4. Evaluation (same modules and code path as training)

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

rep = json.load(open(OUT / "test_report.json"))
names = rep["class_names"]
cm = np.array(rep["confusion_matrix"])
cmn = cm / cm.sum(axis=1, keepdims=True).clip(min=1)

print(f"n={rep['n']} | accuracy {rep['accuracy']:.4f} | UAR {rep['uar']:.4f} | "
      f"macro-F1 {rep['macro_f1']:.4f} | ECE {rep['ece']:.4f}"
      + (f" | held-out corpus: {rep['holdout_dataset']}" if rep["holdout_dataset"] else ""))
display(pd.DataFrame(rep["per_class"]).T.round(3))

fig, ax = plt.subplots(1, 3, figsize=(23, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=names, yticklabels=names, ax=ax[0])
ax[0].set(title="Confusion matrix (counts)", xlabel="Predicted", ylabel="True")
sns.heatmap(cmn, annot=True, fmt=".2f", cmap="YlOrRd", vmin=0, vmax=1,
            xticklabels=names, yticklabels=names, ax=ax[1])
ax[1].set(title="Confusion matrix (row-normalised = recall)", xlabel="Predicted", ylabel="True")

preds = pd.read_csv(OUT / "test_predictions.csv")
preds["correct"] = preds["true"] == preds["pred"]
ax[2].hist(preds.loc[preds.correct, "confidence"], bins=20, alpha=.7, label="correct")
ax[2].hist(preds.loc[~preds.correct, "confidence"], bins=20, alpha=.7, label="wrong")
ax[2].set(title="Confidence: correct vs wrong", xlabel="max softmax prob"); ax[2].legend()
plt.tight_layout()
plt.savefig(WORK / "eval_summary.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Export best checkpoint and verify inference end-to-end

In [ ]:
%%writefile /kaggle/working/emotion_classification/infer.py
# Inference through the SAME modules/yaml used in training (guarantees output_norm etc. match).
import torch
from hyperpyyaml import load_hyperpyyaml
from speechbrain.dataio.encoder import CategoricalEncoder

from ser_common import load_wav


class SERPredictor:
    def __init__(self, hparams_file, overrides=None, device=None):
        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
        with open(hparams_file) as f:
            hp = load_hyperpyyaml(f, overrides or {})
        ckpt = hp["checkpointer"]
        ckpt.recover_if_possible()
        if not ckpt.list_checkpoints():
            raise FileNotFoundError(f"No checkpoint found under {ckpt.checkpoints_dir}")
        self.sr = hp["sample_rate"]
        self.max_len = int(hp["max_eval_sec"] * self.sr)
        self.wavlm = hp["wavlm"].to(self.device).eval()
        self.pooling = hp["pooling"].to(self.device).eval()
        self.classifier = hp["classifier"].to(self.device).eval()
        enc = CategoricalEncoder()
        enc.load(hp["label_encoder_file"])
        self.labels = [enc.ind2lab[i] for i in range(len(enc.ind2lab))]

    @torch.no_grad()
    def predict(self, path, top_k=3):
        wav = load_wav(path, self.sr)[: self.max_len].unsqueeze(0).to(self.device)
        lens = torch.ones(1, device=self.device)
        with torch.autocast("cuda", dtype=torch.float16, enabled=self.device.type == "cuda"):
            feats = self.wavlm(wav)
        pooled = self.pooling(feats.float(), lens)
        probs = self.classifier(pooled).float().softmax(dim=-1).squeeze(0).cpu()
        conf, idx = probs.max(dim=-1)
        top = probs.topk(min(top_k, len(self.labels)))
        return {
            "prediction": self.labels[idx.item()],
            "confidence": round(conf.item(), 4),
            "top_k": [(self.labels[i], round(p, 4)) for p, i in zip(top.values.tolist(), top.indices.tolist())],
            "all_probs": {l: round(float(p), 4) for l, p in zip(self.labels, probs)},
        }

In [ ]:
BEST = WORK / "best_model"
if BEST.exists():
    shutil.rmtree(BEST)

ckpts = sorted(SAVE_FOLDER.glob("CKPT+*"))
assert ckpts, f"No checkpoint in {SAVE_FOLDER}"
best = ckpts[-1]                                  # only the best-by-UAR checkpoint is kept
shutil.copytree(best, BEST / "save" / best.name, ignore=shutil.ignore_patterns("optimizer.ckpt"))
shutil.copy2(SAVE_FOLDER / "label_encoder.txt", BEST / "save")
for f in ("train.yaml", "ser_common.py", "infer.py"):
    shutil.copy2(WORK / f, BEST)
for f in ("test_report.json", "test_predictions.csv"):
    shutil.copy2(OUT / f, BEST)

size_mb = sum(p.stat().st_size for p in BEST.rglob("*") if p.is_file()) / 1e6
print(f"Exported {best.name} -> {BEST}  ({size_mb:.0f} MB, optimizer state excluded)")
print(open(best / "CKPT.yaml").read())

In [ ]:
import glob
sys.path.insert(0, str(WORK))
from infer import SERPredictor

predictor = SERPredictor(str(BEST / "train.yaml"), overrides={"output_folder": str(BEST)})
print("Labels:", predictor.labels)

files = sorted(glob.glob(TEST_AUDIO_GLOB, recursive=True))[:10]
if not files:
    print("No audio matched TEST_AUDIO_GLOB; skipping demo.")
for f in files:
    r = predictor.predict(f)
    print(f"{os.path.basename(f):40s} -> {r['prediction']:<10s} {r['confidence']:.1%}   top3={r['top_k']}")

## Next steps
- Compare `eval_summary.png` per-class recall against the README's Sad/Neutral concern.
- For an honest generalisation estimate, rerun with `HOLDOUT_DATASET = "tess"` (or another corpus) and a new `RUN_TAG`.
- Fit temperature scaling on the validation set (`scripts/calibration_fit.py`) and compare ECE before/after.
- Move `ser_common.py` / `infer.py` logic into `app/model/loader.py` so serving, notebook and CI share one code path.